this is where we run our model and show lots of great results

Assume that we have all of our stuff combined togehter

In [ ]:
# i chat gpt some training code to see if it works :p
import data_utils.data_loader as data_pipeline
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import matplotlib.pyplot as plt
import pandas as pd
import torch

# --- CONFIG ---
JSON_PATH = "data/default_dataset/train.json"
MODEL_NAME = "microsoft/deberta-v3-base"
OUTPUT_DIR = "pii_model_output"
BATCH_SIZE = 4

# 1. GET DATASETS
# We use your custom pipeline to load data
train_dataset, val_dataset, tokenizer = data_pipeline.get_datasets(
    JSON_PATH, MODEL_NAME
)

# 2. MODEL
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(data_pipeline.ID2LABEL),
    id2label=data_pipeline.ID2LABEL,
    label2id=data_pipeline.LABEL2ID,
)

# 3. COLLATOR (Handles padding)
data_collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=16)

# 4. ARGUMENTS
# Key Change: 'logging_steps' controls how often we record the training loss
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",  # Run validation at the end of every epoch
    save_strategy="epoch",  # Save model at the end of every epoch
    logging_strategy="steps",  # Log training loss every X steps
    logging_steps=50,  # Record training loss every 50 batches
    report_to="none",  # Disable external loggers for simplicity
)

# 5. INITIALIZE TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 6. TRAIN
print("Starting Training...")
train_result = trainer.train()

# 7. EXTRACT & PLOT RESULTS
print("Extracting metrics...")

# 'log_history' is a list of dicts.
# Some dicts contain {'loss', 'step'} (Training)
# Others contain {'eval_loss', 'step'} (Validation)
history = trainer.state.log_history

# Convert to DataFrame for easy handling
df = pd.DataFrame(history)

# Separate Train and Validation metrics
train_loss = df[df["loss"].notnull()][["step", "loss", "epoch"]]
val_loss = df[df["eval_loss"].notnull()][["step", "eval_loss", "epoch"]]

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(train_loss["step"], train_loss["loss"], label="Training Loss")
# We plot validation loss at the specific steps it was calculated
plt.plot(val_loss["step"], val_loss["eval_loss"], label="Validation Loss", marker="o")

plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

# Save the plot to a file so you can see it
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
print(f"Loss curve saved to {OUTPUT_DIR}/loss_curve.png")

# 8. SAVE FINAL MODEL
trainer.save_model(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")

AttributeError: 'NoneType' object has no attribute 'endswith'